# Live: Pharmacovigilance Signal Monitor

Polls the public openFDA drug adverse event feed on a fixed interval and
re-renders an interactive Plotly chart each time, so the notebook visibly
refreshes with new data while it runs — a genuinely real-time demo, not a
canned animation. Pharma flavor of "watch a live feed" (openFDA is
continuously updated as new reports are submitted).

Pure Python + pandas + Plotly — no Spark session needed, since this reads
directly from the live API rather than a persisted table. For a durable,
queryable copy see `data-generators/fda_adverse_event_ingest.ipynb`.

Requires `plotly` and `pandas` (see `requirements.txt` in this folder).

In [ ]:
import json
import time
import urllib.request
from datetime import datetime, timezone

import pandas as pd
import plotly.express as px
from IPython.display import clear_output

## Configuration

`N_REFRESHES` bounds the loop so the demo doesn't run forever — bump it
up for a longer-running live dashboard, or re-run the last cell any time
for a single fresh snapshot. Each refresh pulls the most recent batch of
reports sorted by receive date.

In [ ]:
API_URL = "https://api.fda.gov/drug/event.json?sort=receivedate:desc&limit=100"
REFRESH_SECONDS = 60
N_REFRESHES = 5

## Fetch + flatten one snapshot

In [ ]:
def fetch_recent_reactions_df():
    with urllib.request.urlopen(API_URL) as resp:
        data = json.load(resp)

    rows = []
    for report in data.get("results", []):
        patient = report.get("patient", {})
        for reaction in patient.get("reaction", []):
            rows.append({
                "safety_report_id": report.get("safetyreportid"),
                "reaction_term": reaction.get("reactionmeddrapt"),
                "serious": report.get("serious") == "1",
            })
    return pd.DataFrame(rows)

## Live loop

Each iteration fetches a fresh snapshot, clears the previous plot, and
renders the top reaction terms in the current batch.

In [ ]:
for i in range(N_REFRESHES):
    reactions_df = fetch_recent_reactions_df()
    top_reactions_df = (
        reactions_df.groupby("reaction_term")
        .agg(report_count=("safety_report_id", "count"), serious_count=("serious", "sum"))
        .reset_index()
        .sort_values("report_count", ascending=False)
        .head(15)
    )
    refreshed_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    fig = px.bar(
        top_reactions_df.sort_values("report_count"),
        x="report_count",
        y="reaction_term",
        orientation="h",
        color="serious_count",
        color_continuous_scale="OrRd",
        title=f"Live Adverse Event Signals — top reactions in most recent 100 reports — refreshed {refreshed_at}",
    )
    fig.update_layout(template="plotly_dark", height=560, margin=dict(l=0, r=0, t=60, b=0), yaxis_title="")

    clear_output(wait=True)
    fig.show()
    print(f"Refresh {i + 1}/{N_REFRESHES} — {len(reactions_df)} reaction records in this batch")

    if i < N_REFRESHES - 1:
        time.sleep(REFRESH_SECONDS)